# DEC5 000973: depth indexing and shared query gauge

## tl;dr
Integer-depth nearest lookup removes a synthetic plane bias but does not repair the real skin/tube artifact. Completing the recorded BA coordinate transform on query cameras improves F PSNR from 16.915 to 25.986 and LPIPS from .417596 to .071790, but remains worse than the original rig (.047989 LPIPS). Both real controls fail all three native camera anchors.

## Context & Methods
Two isolated controls at **one temporal frame**, not six temporal samples. The depth control changes only VBG depth lookup. The gauge control keeps the BA mesh/train calibration/RGB fixed and transforms untouched query poses using the existing train-only BA receipt. No held RGB determines either transformation.

### Key Assumptions
The pinned COLMAP MVS code uses integer depth rays. Nearest lookup matches that producer convention; it does not prove physical RGB-center correctness. Query-gauge completion is a coordinate operation, not a new held-camera calibration fit. The fixed study face ROI excludes most neck/hand, so native review remains mandatory.

## Data
Read-only artifacts under `/mnt/data/lookcloser_dec5_5a3_surface_repair`; run on clever-shadow with repository `.venv`, NumPy and Pillow. The experiment report contains pinned upstream-code references and native comparisons. No reconstruction runs or output images are altered here.


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
from PIL import Image

root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair')
families = ['integer_depth_convention/real_000973', 'complete_gauge_control']
def read(path):
    return json.loads(Path(path).read_text())
def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

findings = {}
for family in families:
    result = read(root/family/'findings.json')
    assert result['audit_status'] == 'comparison_integrity_pass_candidate_quality_fail'
    assert result['temporal_frame_count'] == 1
    assert result['visual_pass'] == 0 and result['visual_fail'] == 3
    assert not result['temporal_promotion']
    for path, expected in result['retained_hashes'].items():
        assert sha(path) == expected, path
    findings[family] = result
    print(family, 'retained hashes verified:', len(result['retained_hashes']))


integer_depth_convention/real_000973 retained hashes verified: 195
complete_gauge_control retained hashes verified: 144


## Results

### Installed synthetic CPU/CUDA plane control
Residuals are distances in synthetic world units, not measured scene millimeters.


In [2]:
synthetic = read(root/'integer_depth_convention/synthetic/findings.json')
assert len(synthetic['rows']) == 4
by_key = {(r['device'], r['sampling']): r for r in synthetic['rows']}
assert len(by_key) == 4
for device in ['CPU:0', 'CUDA:0']:
    old = by_key[(device, 'legacy_floor')]
    new = by_key[(device, 'nearest_integer')]
    assert old['mean_plane_residual'] > .001
    assert abs(new['mean_plane_residual']) < .0002
    assert new['plane_residual_rmse'] < .7*old['plane_residual_rmse']
    print(device, 'mean signed residual:', old['mean_plane_residual'], '->', new['mean_plane_residual'])
    print(device, 'RMSE:', old['plane_residual_rmse'], '->', new['plane_residual_rmse'])
for mode in ['legacy_floor', 'nearest_integer']:
    for key in ['admitted_vertices', 'mean_plane_residual', 'plane_residual_rmse']:
        assert by_key[('CPU:0', mode)][key] == by_key[('CUDA:0', mode)][key]
integer = findings[families[0]]
assert integer['depth_maps_verified'] == 62 and integer['mesh_components'] == 1
assert integer['mesh_vertices'] == 80193 and integer['mesh_triangles'] == 154966
print('Real control mesh:', integer['mesh_vertices'], 'vertices;', integer['mesh_triangles'], 'triangles')
print('Unchanged depth coverage mean/min:', integer['depth_coverage_mean'], integer['depth_coverage_min'])


CPU:0 mean signed residual: 0.001439298464883445 -> -3.394553763341263e-05
CPU:0 RMSE: 0.0015623670783236228 -> 0.0006009483131528512
CUDA:0 mean signed residual: 0.001439298464883445 -> -3.394553763341263e-05
CUDA:0 RMSE: 0.0015623670783236228 -> 0.0006009483131528512
Real control mesh: 80193 vertices; 154966 triangles
Unchanged depth coverage mean/min: 0.38273008917015133 0.24788242669753086


### Same GT/ROI, explicit comparison baselines
The BA comparison uses the same mesh and trained camera parameters; query coordinates differ. The depth comparison instead keeps the original camera rig.


In [3]:
metric_keys = ['face_psnr', 'face_ssim', 'face_lpips']
paths = {
    'original_rig': root/'depth_separated_source_control/off_rank001/face_metrics/metrics.json',
    'nearest_integer_depth': root/families[0]/'face_metrics/metrics.json',
    'BA_old_query_gauge': root/'calibration_control/reconstruct_000973/face_metrics/metrics.json',
    'BA_common_query_gauge': root/families[1]/'face_metrics/metrics.json',
}
baseline = read(paths['original_rig'])
metrics = {}
for name, path in paths.items():
    m = read(path)
    for key in ['ground_truth_sha256', 'face_polygons_sha256', 'face_bbox_xyxy', 'protocol', 'metric_domain']:
        assert m[key] == baseline[key]
    for key in ['prediction', 'ground_truth', 'face_polygons']:
        assert sha(m[key]) == m[key+'_sha256']
    assert all(np.isfinite(m[key]) for key in metric_keys)
    assert not any(key.startswith(('loss', 'full_frame', 'room_psnr', 'actor_psnr')) for key in m)
    metrics[name] = m
    print(name, {key: round(m[key], 6) for key in metric_keys})
gain = metrics['BA_common_query_gauge']['face_psnr'] - metrics['BA_old_query_gauge']['face_psnr']
print('BA query-gauge PSNR recovery, dB:', round(gain, 6))
assert metrics['BA_common_query_gauge']['face_lpips'] > baseline['face_lpips']


original_rig {'face_psnr': 28.724285, 'face_ssim': 0.889098, 'face_lpips': 0.047989}
nearest_integer_depth {'face_psnr': 28.694189, 'face_ssim': 0.887771, 'face_lpips': 0.048337}
BA_old_query_gauge {'face_psnr': 16.915396, 'face_ssim': 0.655352, 'face_lpips': 0.417596}
BA_common_query_gauge {'face_psnr': 25.986305, 'face_ssim': 0.816415, 'face_lpips': 0.07179}
BA query-gauge PSNR recovery, dB: 9.07091


### Gauge propagation and exact old-render replays
No query RGB enters the calibration operation. Applying one similarity to a camera and a scene point must preserve normalized projection.


In [4]:
gauge_root = root/'complete_gauge_control'
request = read(gauge_root/'request.json')
gauge = request['gauge']
rotation = np.asarray(gauge['rotation'])
scale = gauge['scale']
translation = np.asarray(gauge['translation'])
original = read('/mnt/data/lookcloser_dec5_5a3_patchmatch_tsdf_50/config/calibration/transforms.json')
refined = read(root/'calibration_control/rig_focal_regularized/transforms.json')
completed = read(gauge_root/'calibration.json')
query = {'F004_B005_1210O9', 'J004_D005_1210TA', 'L004_B005_12106A'}
before = {f['physical_camera']: f for f in original['frames']}
old_rows = {f['physical_camera']: f for f in refined['frames']}
new_rows = {f['physical_camera']: f for f in completed['frames']}
assert len(old_rows) == len(new_rows) == 65
for name, old in old_rows.items():
    new = new_rows[name]
    if name not in query:
        assert old == new
        continue
    assert old == before[name]
    old_pose = np.asarray(old['transform_matrix'])
    new_pose = np.asarray(new['transform_matrix'])
    np.testing.assert_allclose(new_pose[:3, :3], rotation@old_pose[:3, :3], atol=1e-12)
    np.testing.assert_allclose(new_pose[:3, 3], scale*(rotation@old_pose[:3, 3])+translation, atol=1e-12)
    assert {k:v for k,v in old.items() if k!='transform_matrix'} == {k:v for k,v in new.items() if k!='transform_matrix'}
    point = old_pose[:3, 3] + old_pose[:3, :3]@np.array([.01, -.02, -.7])
    transformed = scale*(rotation@point)+translation
    old_camera = (point-old_pose[:3, 3])@old_pose[:3, :3]
    new_camera = (transformed-new_pose[:3, 3])@new_pose[:3, :3]
    np.testing.assert_allclose(new_camera/new_camera[2], old_camera/old_camera[2], atol=1e-12)
for index in range(3):
    assert sha(gauge_root/'legacy_replay'/f'view_{index:04d}.png') == sha(root/'calibration_control/reconstruct_000973/three_views'/f'view_{index:04d}.png')
assert findings[families[1]]['same_mesh_and_train_cameras']
print('62 train camera rows unchanged; 3 queries receive the exact recorded gauge.')
print('Three old BA PNGs replay byte-for-byte; normalized query projection invariant.')
print('Recorded gauge rotation, degrees:', np.degrees(np.arccos(np.clip((np.trace(rotation)-1)/2, -1, 1))))


62 train camera rows unchanged; 3 queries receive the exact recorded gauge.
Three old BA PNGs replay byte-for-byte; normalized query projection invariant.
Recorded gauge rotation, degrees: 0.24874482132570996


### Native crop inventory and verdicts
These checks validate saved review evidence, not replace LLM visual judgment.


In [5]:
all_crops = []
for family in families:
    review = read(root/family/'visual_review.json')
    crops = [path for row in review['rows'] for path in row['inspected_crops']]
    assert len(crops) == len(set(crops)) == 7
    assert all(row['visual_status'] == 'fail' for row in review['rows'])
    expected_panels = 3 if family.startswith('integer') else 4
    for path in crops:
        path = Path(path)
        source = read(path.parent/'review_inputs.json')
        assert source['rotate'] == 90 and len(source['images']) == expected_panels
        box = next(s for s in source['boxes'] if s.split(':')[0] == path.stem)
        x0, y0, x1, y1 = map(int, box.split(':')[1].split(','))
        with Image.open(path) as image:
            assert image.size == (expected_panels*(y1-y0), x1-x0+28)
        for image in source['images']:
            assert sha(image['path']) == image['sha256']
    all_crops.extend(crops)
print('14 actual native comparison panels verified; 0/6 accepted real variant-anchor outputs.')


14 actual native comparison panels verified; 0/6 accepted real variant-anchor outputs.


## Takeaways
- A producer/consumer indexing mismatch explains the synthetic plane bias, not the persistent real skin patch.
- The earlier BA's large render regression was materially confounded by query-gauge inconsistency. The recorded train-only transform recovers 9.07 dB without changing the mesh or fitting held RGB.
- The common-gauge BA still fails native review and remains below the original rig in face quality. Gauge consistency is necessary for interpretable tests, not sufficient for a correct skin/tube surface.
- Neither result is promoted to every-40th-available-frame or continuous fly-through validation. The original campaign and production defaults remain unchanged.
- [Experiment report](../dec5_patchmatch_surface_repair.md) contains the upstream sources, artifact links and limitations.
